In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'IMDB'
name_model = 'roberta-base'
seed = 3
part = 4

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post8/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post9/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/cloudrun-invoker-only.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
import google.auth.transport.requests
from google.oauth2 import service_account

In [10]:
SERVICE_URL = f"https://{name_dataset.replace('_', '-').lower()}-{name_model}-seed-{seed}-272427236032.us-central1.run.app"

In [11]:
credentials = service_account.IDTokenCredentials.from_service_account_file(
    path_credentials,
    target_audience=SERVICE_URL)

auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)

In [12]:
def predict_label(text):

  payload = {"text": text}

  headers = {
      "Authorization": f"Bearer {credentials.token}",
      "Content-Type": "application/json"
  }

  start = time.perf_counter()

  resp = requests.post(f"{SERVICE_URL}/", headers=headers, json=payload, timeout=60)

  end = time.perf_counter()

  latency_ms = (end - start) * 1000

  output = {
      'label': int(json.loads(resp.text)['prediction']),
      'latency': latency_ms
  }

  return output

# 3. Load Dataset

In [13]:
df = pd.read_csv(path_open)

In [14]:
df.shape

(2500, 19)

In [15]:
pred_label = []
pred_latency = []

In [16]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  output = predict_label(text)

  pred_label.append(output['label'])
  pred_latency.append(output['latency'])

  if (i % 10) == 0:
    print(i)

0


10


20


30


40


...

2450


2460


2470


2480


2490


In [17]:
df[f'{name_model}-seed-{seed}-label'] = pred_label
df[f'{name_model}-seed-{seed}-latency'] = pred_latency

In [18]:
df[f'{name_model}-seed-{seed}-label'].value_counts()

,count
roberta-base-seed-3-label,
0,1252
1,1248


In [19]:
df[f'{name_model}-seed-{seed}-latency'].describe()

,roberta-base-seed-3-latency
count,2500.000000
mean,704.294435
std,355.257879
min,213.197017
25%,444.863975
50%,583.734872
75%,902.832708
max,2359.652424


# 4. Save Dataset

In [20]:
df.to_csv(path_save, index = False)

# 5. Execution Time

In [21]:
end_notebook = time.time()

In [22]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 31m 23.89s
